# demo_multiagent

> The multi-agent write-up (plan deliverable, Phase 10 decision L20): the headline question answered from the saved results of Phases 8-10, without rerunning anything.

**Headline question.** At a fixed total compute budget, how does team performance change with team size under different communication harnesses, and how robust is each harness when one agent lies? **Hypothesis** (from the fly paper: a stable internal map, only the goal moves): learned agents approach the ideal free distribution faster with a shared map, at the cost of less belief diversity and more vulnerability to liars.

This notebook reads `results/multiagent/{bench/full, safety/h3, safety/full, llm/full}/` (gitignored; rebuild them with the Phase 8, 9 and 10 `RUN.md` commands) and writes `results/multiagent/REPORT.md` and its figures to `results/multiagent/report/`. If the saved results are missing, it says so and writes nothing.

In [ ]:
import json
import math
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

from explRL.bench_grid import readCSV
from explRL.bench_stats import bootCI
from explRL.followup_report import summaries as followupSummaries
from explRL.llm_report import summaries
from explRL.team_time import marginalGain
from explRL.train import demo_artifact_dir

R = demo_artifact_dir(None, 'multiagent')
NEED = [R / 'bench/full/metrics.json', R / 'bench/full/runs.csv', R / 'safety/h3/metrics.json',
        R / 'safety/full/metrics.json', R / 'llm/full/runs.csv', R / 'llm/full/metrics.json']
HAVE = all(p.exists() for p in NEED)
print('saved results found' if HAVE else 'missing: ' + ', '.join(str(p.relative_to(R)) for p in NEED if not p.exists()))
COLORS = dict(none='tab:gray', blackboard='tab:blue', messages='tab:orange', hierarchical='tab:green')
CH = tuple(COLORS)


def ci(x):
    x = np.asarray([v for v in x if not math.isnan(v)], float)
    return float(x.mean()), *bootCI(x)


def fmt(m, lo, hi, d=3):
    return f'{m:.{d}f} [{lo:.{d}f}, {hi:.{d}f}]'

In [ ]:
if HAVE:
    bench = json.loads((R / 'bench/full/metrics.json').read_text())
    h3 = json.loads((R / 'safety/h3/metrics.json').read_text())
    safety = json.loads((R / 'safety/full/metrics.json').read_text())
    llmMeta = json.loads((R / 'llm/full/metrics.json').read_text())
    benchRows = [r for r in readCSV(R / 'bench/full/runs.csv') if r['study'] == 'main']
    llmRows = readCSV(R / 'llm/full/runs.csv')
    S10 = summaries([r for r in llmRows if r['runner'] == 'llm'], [r for r in llmRows if r['runner'] == 'numpy'])
    out = R / 'report'
    out.mkdir(exist_ok=True)
    opt = lambda q: readCSV(R / q) if (R / q).exists() else []
    teamRate8 = opt('bench/teamrate/runs.csv')                  # Phase 8 main grid rerun with team rates (post hoc)
    followRows = opt('followup/full/runs.csv') + opt('followup/numpy/runs.csv') + opt('followup/pilot/runs.csv')
    SF = followupSummaries(followRows, llmRows) if followRows else None

## Figures

In [ ]:
def scalingFig(path):
    "Phase 8: welfare per agent-second vs team size per channel, fixed compute and fixed world time."
    fig, axs = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
    for ax, b in zip(axs, ('compute', 'time')):
        for ch in CH:
            Ns = sorted({int(r['N']) for r in benchRows})
            st = [ci([r['perAgentSecond'] for r in benchRows if r['channel'] == ch and r['budget'] == b and int(r['N']) == N]) for N in Ns]
            m = np.array([s[0] for s in st])
            ax.errorbar(Ns, m, yerr=[m - [s[1] for s in st], [s[2] for s in st] - m], label=ch, color=COLORS[ch], marker='o', capsize=2)
        ax.set(xscale='log', xlabel='team size N', title=f'fixed {"compute (N x T constant)" if b == "compute" else "world time"}')
        ax.set_xticks(Ns, [str(N) for N in Ns]); ax.minorticks_off()
    axs[0].set_ylabel('welfare per agent-second'); axs[0].legend(fontsize=8)
    fig.suptitle('Phase 8: numpy bandit teams (30 seeds)'); fig.tight_layout(); fig.savefig(path, dpi=110); plt.close(fig)


def robustnessFig(path):
    "Phase 9: honest agents' relative welfare loss vs share of liars, per channel, N = 16 and 64."
    loss = safety['tables']['attackLoss']
    fig, axs = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
    for ax, N in zip(axs, (16, 64)):
        for ch in CH:
            pts = sorted((int(k.split('/')[3]) / N, v['loss']) for k, v in loss.items()
                         if k.startswith(f'{ch}/{N}/liar/'))
            if pts:
                x, m = [0.0] + [p[0] for p in pts], np.array([0.0] + [p[1][0] for p in pts])
                lo, hi = np.array([0.0] + [p[1][1] for p in pts]), np.array([0.0] + [p[1][2] for p in pts])
                ax.errorbar(x, m, yerr=[m - lo, hi - m], label=ch, color=COLORS[ch], marker='o', capsize=2)
        ax.set(xlabel='share of liars in the team', title=f'N = {N}'); ax.axhline(0, color='k', lw=0.5)
    axs[0].set_ylabel("honest agents' welfare loss (share)"); axs[0].legend(fontsize=8)
    fig.suptitle('Phase 9: robustness to liars (10 seeds)'); fig.tight_layout(); fig.savefig(path, dpi=110); plt.close(fig)


def llmFig(path):
    "Phase 10: welfare per agent-turn vs team size per channel, task board and text foraging (Claude Haiku 5.5)."
    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    for ax, task in zip(axs, ('board', 'forage')):
        for ch in CH:
            pts = sorted((N, s) for (t, c, N), s in S10['main'].items() if t == task and c == ch)
            m = np.array([p[1]['mean'] for p in pts])
            err = [m - [p[1]['ci'][0] for p in pts], [p[1]['ci'][1] for p in pts] - m]
            ax.errorbar([p[0] for p in pts], m, yerr=err, label=ch, color=COLORS[ch], marker='o', capsize=2)
        rho = S10['numpy'][task]
        ax.set(xscale='log', xlabel='team size N', ylabel='welfare per agent-turn',
               title=f"{'task board' if task == 'board' else 'text foraging'} (numpy-LLM rho {rho['rho']:.2f})")
        ax.set_xticks([1, 2, 4, 8], ['1', '2', '4', '8']); ax.minorticks_off(); ax.axhline(0, color='k', lw=0.5)
    axs[0].legend(fontsize=8)
    fig.suptitle('Phase 10: Claude Haiku 5.5 teams (5 seeds, 50 turns per agent)'); fig.tight_layout(); fig.savefig(path, dpi=110)
    plt.close(fig)


def teamFig(path):
    "Team totals (post hoc): Phase 8 team total vs N (both budgets), Phase 8 cumulative harvest over time, Phase 10 team totals."
    fig, axs = plt.subplots(1, 3, figsize=(14, 4))
    Ns = sorted({int(r['N']) for r in benchRows})
    for ch in CH:
        for b, ls in (('compute', '-'), ('time', '--')):
            m = [ci([r['welfare'] for r in benchRows if r['channel'] == ch and r['budget'] == b and int(r['N']) == N])[0] for N in Ns]
            axs[0].plot(Ns, m, ls, color=COLORS[ch], marker='o', ms=3, label=ch if b == 'compute' else None)
        rs = [r for r in teamRate8 if r['channel'] == ch and r['budget'] == 'time' and int(r['N']) == 16]
        K = min((sum(k.startswith('teamRate_') and r[k] == r[k] for k in r) for r in rs), default=0)
        if K:
            axs[1].plot(40 * np.arange(1, K + 1), np.mean([[r[f'teamRate_{k}'] * 40 for k in range(K)] for r in rs], 0).cumsum(),
                        color=COLORS[ch], marker='.', label=ch)
        for task, ls in (('board', '-'), ('forage', '--')):
            pts = sorted((N, s['team']['mean']) for (t, c, N), s in S10['main'].items() if t == task and c == ch)
            axs[2].plot([x for x, _ in pts], [y for _, y in pts], ls, color=COLORS[ch], marker='o', ms=3)
    axs[0].set(xscale='log', xlabel='team size N', ylabel='team total (net harvest)', title='Phase 8 team total')
    axs[0].set_xticks(Ns, [str(N) for N in Ns]); axs[0].minorticks_off(); axs[0].legend(fontsize=7)
    axs[1].set(xlabel='world time (s)', ylabel='team harvest, cumulative', title='Phase 8 harvest over time (N = 16)')
    axs[1].legend(fontsize=7)
    axs[2].set(xscale='log', xlabel='team size N', ylabel='team total', title='Phase 10 team total')
    axs[2].set_xticks([1, 2, 4, 8], ['1', '2', '4', '8']); axs[2].minorticks_off()
    fig.suptitle('Team totals (post hoc). Left: solid = fixed compute, dashed = fixed world time. Right: solid = task board, dashed = text foraging', fontsize=10)
    fig.tight_layout(); fig.savefig(path, dpi=110); plt.close(fig)


if HAVE:
    scalingFig(out / 'scaling.png'); robustnessFig(out / 'robustness.png'); llmFig(out / 'llm.png')
    if teamRate8:
        teamFig(out / 'team.png')

## Report

In [ ]:
def teamSection():
    "Section 5: team totals (post hoc), from the saved rows."
    if not teamRate8:
        return []
    tt = lambda ch, b, N: ci([r['welfare'] for r in benchRows if r['channel'] == ch and r['budget'] == b and int(r['N']) == N])
    g = marginalGain([r for r in llmRows if r['study'] == 'main' and r['runner'] == 'llm'])
    return ['## 5. Team totals (post hoc)', '', '![team](report/team.png)', '',
            'Per-agent welfare must fall when patches have limited capacity; the team total shows whether more agents still add '
            'food. At fixed compute the two rank teams identically (the total is per-agent welfare times a constant). At fixed '
            f"world time (Phase 8) the blackboard team's total grows from {fmt(*tt('blackboard', 'time', 1), d=0)} with one agent to "
            f"{fmt(*tt('blackboard', 'time', 16), d=0)} at N = 16 and {fmt(*tt('blackboard', 'time', 64), d=0)} at N = 64, against "
            f"{fmt(*tt('none', 'time', 64), d=0)} without a channel at N = 64. On the Phase 10 task board each added agent still adds "
            f"value with the claim board (marginal gain {g[('board', 'blackboard', 8)]:.1f} per added agent from 4 to 8) but much "
            f"less without one ({g[('board', 'none', 8)]:.1f}).", '']


def followSection():
    "Section 6: the follow-up runs after Phase 10, when saved."
    if SF is None:
        return []
    e, n, l = SF['effort'], SF['numpy'], SF['llm']
    L = ['## 6. Follow-up runs after Phase 10 (exploratory)', '',
         '- **Hard board (D1):** paused. Claude Haiku 5.5 solved 91-98% of the puzzles it opened even at difficulty 10 and 12, so '
         'no level reached the planned 30-70% band; the puzzles are to be redesigned.']
    if e.get('Sonnet minus Haiku, effort medium', {}).get('n'):
        L.append(f"- **Model size at matched effort (D2):** Sonnet's blackboard benefit minus Haiku's at N = 4: effort low "
                 f"{fmt(e['Sonnet minus Haiku, effort low']['mean'], *e['Sonnet minus Haiku, effort low']['ci'], d=2)}, effort medium "
                 f"{fmt(e['Sonnet minus Haiku, effort medium']['mean'], *e['Sonnet minus Haiku, effort medium']['ci'], d=2)}.")
    if n['cells']:
        best = n['diff'].get((8, 'squads|beta=inf'))
        L.append(f"- **Rounds and soft territories, numpy (B-numpy):** at N = 8, squads that stay home earn "
                 f"{fmt(best['mean'], *best['ci'], d=3)} more per agent-second than the event-driven team; synchronous rounds alone "
                 f"cost welfare at every size, and at 32-64 agents no squad arm comes back to the event-driven level.")
    if l['diff'].get('rounds - events: welfare', {}).get('n'):
        sq = [k for k in l['diff'] if k.startswith('squads') and k.endswith(': welfare')]
        L.append(f"- **Rounds and squads, LLM (B-LLM, N = 8):** rounds minus event-driven "
                 f"{fmt(l['diff']['rounds - events: welfare']['mean'], *l['diff']['rounds - events: welfare']['ci'], d=2)}" +
                 (f"; squads minus rounds {fmt(l['diff'][sq[0]]['mean'], *l['diff'][sq[0]]['ci'], d=2)}" if sq else '') +
                 ' (welfare per agent-turn).')
    return L + ['', 'Details: `specs/multiagent/speedup/README.md`, `results/multiagent/followup/*/report.md`.', '']


def report():
    c, t = bench['confirmatory'], h3['h3']['H3']
    sc = {(ch, b): ci([r['perAgentSecond'] for r in benchRows if r['channel'] == ch and r['budget'] == b and int(r['N']) == N])
          for ch in CH for b in ('compute', 'time') for N in (64,)}
    l64 = safety['tables']['attackLoss']
    b10, ben = S10['main'], S10['benefit']
    sz, npy = S10['sizeContrast'], S10['numpy']
    row = lambda name, v, unit='': f"| {name} | {fmt(v['mean'], *v['ci'])}{unit} | {v['pHolm']:.4f} | **{v['verdict']}** |"
    L = ['# Multi-agent harnesses: results report (Phases 5-10)', '',
         'Generated by `nbs/demo_multiagent.ipynb` from the saved results (no rerun). Specs: `specs/multiagent/`.', '',
         '## Question and answer', '',
         '> At a fixed total compute budget, how does team performance change with team size under different communication '
         'harnesses, and how robust is each harness when one agent lies?', '',
         f"**Answer (numpy agents, Phases 8-9; confirmatory).** With a shared map (blackboard), output per agent holds up as the team "
         f"grows at fixed compute ({fmt(*sc[('blackboard', 'compute')])} welfare per agent-second at N = 64, vs "
         f"{fmt(*sc[('none', 'compute')])} without a channel); teams reach the ideal free distribution faster and agree more; "
         f"and the same shared map makes them more vulnerable to a liar than peer messages are. All four pre-registered tests "
         f"were supported. **With language-model agents (Phase 10; exploratory)** the picture splits by task: on a task board where "
         f"the risk is duplicate work, a shared claim board keeps welfare per agent-turn flat from 1 to 8 agents while teams "
         f"without one lose two-thirds of it, and the numpy model predicts the ranking of harnesses (rho {npy['board']['rho']:.2f}); "
         f"in text foraging no harness helps reliably and the numpy model does not predict the LLM teams (rho "
         f"{npy['forage']['rho']:.2f}).", '',
         '## 1. Pre-registered tests', '',
         '| Test (blackboard minus no channel, or minus messages for H3) | Mean [95% CI] | Holm p | Verdict |', '|---|---|---|---|',
         row('P: welfare per agent-second, N 2-64, fixed compute (Phase 8)', c['P']),
         row('H1: recovery time to the ideal free distribution, s, N 4-64 (Phase 8)', c['H1']),
         row('H2: belief diversity, N 4-64 (Phase 8)', c['H2']),
         row("H3: honest agents' loss from one liar, blackboard minus messages, N 8-32 (Phase 9)", t), '',
         f"30 seeds each, run once on frozen pre-registrations (Phase 8 `PREREG` hash `{bench['preregHash']}`, Phase 9 "
         f"`PREREG_H3` hash `{h3['preregHash']}`).", '',
         '## 2. Scaling per harness (Phase 8)', '', '![scaling](report/scaling.png)', '',
         'At fixed compute (left) every team gets the same total agent-time, so bigger teams run for less time each. Without '
         'a channel, or with messages, output per agent collapses as the team grows; the blackboard holds up; the lead starts '
         'worst and improves with size. At fixed world time (right) the blackboard gains from size up to 16 agents.', '',
         '## 3. Robustness to liars (Phase 9)', '', '![robustness](report/robustness.png)', '',
         "Loss of honest agents' welfare as the share of liars grows. On the blackboard and with the lead, liars do real damage "
         f"(a quarter of the team lying costs {l64['blackboard/64/liar/16']['loss'][0]:.0%} on the blackboard at N = 64); through "
         f"messages and with no channel, nothing measurable. Private trust weighting was the only defence that paid; shared "
         f"reputation and sanctions cost more in monitoring than they saved.", '',
         '## 4. Language-model agents (Phase 10, exploratory)', '', '![llm](report/llm.png)', '',
         f"Claude Haiku 5.5 teams of 1-8, 50 turns per agent, 5 seeds; {llmMeta['calls']} API calls, "
         f"${llmMeta['spentTotal']:.2f} in all (cap ${llmMeta['cap']:.0f}); analysis plan hash `{llmMeta['planHash']}`.", '',
         f"- **Task board:** without a channel, welfare per agent-turn falls from {fmt(b10[('board', 'none', 1)]['mean'], *b10[('board', 'none', 1)]['ci'], d=2)} "
         f"(alone) to {fmt(b10[('board', 'none', 8)]['mean'], *b10[('board', 'none', 8)]['ci'], d=2)} at N = 8 (duplicate work); "
         f"the claim blackboard's benefit grows with size (+{ben[('board', 'blackboard', 2)]['mean']:.2f}, "
         f"+{ben[('board', 'blackboard', 4)]['mean']:.2f}, +{ben[('board', 'blackboard', 8)]['mean']:.2f} at N = 2, 4, 8).",
         f"- **Text foraging:** no harness helps reliably; at N = 8 the blackboard costs "
         f"{fmt(ben[('forage', 'blackboard', 8)]['mean'], *ben[('forage', 'blackboard', 8)]['ci'], d=2)}, with less exploration and "
         f"lower belief diversity: the herding side of H2.",
         f"- **Numpy predicts LLM?** Spearman rho over the 16 (harness, size) cells: board {npy['board']['rho']:.2f} "
         f"[{npy['board']['ci'][0]:.2f}, {npy['board']['ci'][1]:.2f}], foraging {npy['forage']['rho']:.2f} "
         f"[{npy['forage']['ci'][0]:.2f}, {npy['forage']['ci'][1]:.2f}].",
         f"- **Stronger model (Claude Sonnet 5.5, board, N 1 and 4):** harness benefit minus Haiku's: blackboard "
         f"{fmt(sz['blackboard']['mean'], *sz['blackboard']['ci'], d=2)}, messages {fmt(sz['messages']['mean'], *sz['messages']['ci'], d=2)}, "
         f"lead {fmt(sz['hierarchical']['mean'], *sz['hierarchical']['ci'], d=2)}: smaller at N = 4 but unclear overall, and "
         f"confounded by Sonnet's many more wrong answers at low effort.",
         '- **Tokens as compute:** at an equal budget of 200 agent-turns, two agents did best; splitting further did not pay. '
         'Prompt-only changes (team or credit framing, persistent notes) did nothing clear; notes-only memory hurt.', '',
         *teamSection(), *followSection(),
         '## Limitations', '',
         '- The numpy results come from a small, abstract world; the LLM results are exploratory, with 3-5 seeds per cell and '
         'one sampled reply per request (reruns are exact only through the response cache).',
         '- The task board turned out easy for Claude Haiku 5.5 (solve rate 0.89 at the hardest usable difficulty), so it tests '
         'coordination against duplicate work, not division of hard labour.',
         '- The free-riding index does not flag free-riders as defined (Phase 9).', '',
         '## Where to look', '',
         '- Phase specs and results: `specs/multiagent/phase5/` to `phase10/` (each with a plain-language figure guide).',
         '- Full per-phase reports: `results/multiagent/bench/full/report.md`, `safety/h3/report.md`, `safety/full/report.md`, '
         '`llm/full/report.md`.']
    return '\n'.join(L) + '\n'


if HAVE:
    text = report()
    (R / 'REPORT.md').write_text(text)
    for s in ('supported',):
        assert text.count(f'**{s}**') == 4
    print(text[:3000])